# Notebook 06: Preserve the input, adapt the architecture, reduce overfitting

Advisor feedback (11 Nov 2026):
1. **Preserve the input.** The images are only 40 × 24 pixels, so every resize changes them a lot. Adapt the **model / architecture** to the input instead of the input to the model, or use another model taken from published work.
2. **Bring the validation loss down** (less overfitting). In R10 the validation loss was lowest at epoch 9 (0.67) and then rose to 0.73, while the training loss fell to 0.04.

### What changes compared with Notebook 05
| | Notebook 05 (R10) | Notebook 06 |
|---|---|---|
| Input | 40 × 24 → resized to 160 × 96, gray repeated to 3 channels | **40 × 24 as is, 1 channel** (only the per-image z-score) |
| Model | standard ResNet-18 (stem made for 224 × 224) | **architectures made for small images** (below) |
| Overfitting control | AdamW wd 1e-4, flips + shift | adds a **regularized recipe** (label smoothing, dropout, wd 5e-4, noise) |
| Overfitting measured by | validation loss only | validation loss **and** loss on a fixed training subset, both plain cross-entropy, so the train/validation gap is measured on equal terms |

### The six architectures (all take 40 × 24 × 1 directly)
| Name | Architecture | Source | Weights |
|---|---|---|---|
| `ResNet18S` | ResNet-18 with the small-image stem: conv1 3×3 stride 1, **no max-pool** (40×24 → … → 5×3) | He et al. 2016, CIFAR-10 setting | ImageNet for layer1–4 + fc removed; conv1 initialised from the centre 3×3 of the pretrained 7×7 filters (RGB summed to 1 channel) |
| `ResNet18S_scratch` | same network | same | random (checks whether ImageNet still helps) |
| `WRN16_4` | Wide ResNet, depth 16, width ×4, made for 32 × 32 images | Zagoruyko & Komodakis 2016 (BMVC) | random |
| `ResNet20` | ResNet-20, the ResNet designed for 32 × 32 images (widths 16/32/64, 0.27 M parameters) | He et al. 2016, CIFAR-10 | random |
| `DenseNetBC100` | DenseNet-BC, L = 100, k = 12 (0.8 M parameters): every layer reuses all earlier feature maps | Huang et al. 2017; DenseNet-type models were used on this dataset (Coupled UDenseNet, *Sensors* 2023) | random |
| `MSCNN` | multi-scale CNN: three parallel branches with 3×3, 5×5, 7×7 kernels, then a shared body | idea from Korkmaz & Açıkgöz 2022 (same dataset); our simplified re-implementation, not their exact network | random |

### The two training recipes
| | `base` (same as R10) | `reg` (regularized) |
|---|---|---|
| Label smoothing | 0 | 0.1 |
| Dropout | 0 | 0.3 (before the last layer; inside the blocks for WRN) |
| Weight decay (AdamW) | 1e-4 | 5e-4 |
| Augmentation | H/V flip p=0.5, shift ±2 px | same + Gaussian noise σ = 0.05 (in z-score units) |

Everything else is fixed: `splits.csv`, sampler p ∝ 1/√n (R10), AdamW + cosine schedule, batch 128, model kept at the epoch with the best validation macro-F1, 3 seeds (42, 1, 2).
Random erasing is **not** used: erasing a 4 × 4 patch can remove the only hot cell and turn a `Cell` image into a normal one.

### Run plan: 6 models × 2 recipes × 3 seeds = 36 runs (R26–R61), roughly 1.5–2.5 hours on the RTX 4070 Ti
Reference for comparison: R10, R16, R17 (Notebook 05: ResNet-18 with resize to 160 × 96, recipe `base`).

**How to run:** Run All. Section 2 is a quick self-test (no logging). Each run is logged exactly like Notebook 05 (`experiment_log.xlsx`, `runs/<RunID>_<time>/`), re-runs never overwrite. **Close `experiment_log.xlsx` before running.**

## 0. Imports, device, data

In [ ]:
import time, datetime, platform, json, sys, hashlib
from pathlib import Path
from copy import copy
import numpy as np
import pandas as pd
from PIL import Image
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler
import torchvision
from torchvision.models import resnet18, ResNet18_Weights
from sklearn.metrics import (f1_score, accuracy_score, confusion_matrix, ConfusionMatrixDisplay,
                             precision_recall_fscore_support)
from openpyxl import load_workbook

NOTEBOOK = "06_preserve_input.ipynb"
REL = Path("2020-02-14_InfraredSolarModules/InfraredSolarModules")
CANDIDATES = [Path("../archive"), Path("../data/archive"), Path("archive"), Path("../../archive")]
DATA_DIR = next((r / REL for r in CANDIDATES if (r / REL / "images").exists()), None)
if DATA_DIR is None:
    raise FileNotFoundError(f"Data folder not found. This notebook runs in {Path.cwd()}\nLooked in:\n  "
                            + "\n  ".join(str((r / REL).resolve()) for r in CANDIDATES))

if torch.cuda.is_available():
    device = torch.device("cuda"); HARDWARE = torch.cuda.get_device_name(0)
    torch.backends.cudnn.benchmark = True
elif torch.backends.mps.is_available():
    device = torch.device("mps"); HARDWARE = "Apple GPU (mps)"
else:
    device = torch.device("cpu"); HARDWARE = platform.processor() or "CPU"

for d in ("figures", "checkpoints", "runs"):
    Path(d).mkdir(exist_ok=True)
print("PyTorch", torch.__version__, "| torchvision", torchvision.__version__)
print("device:", device, "|", HARDWARE)
print("data:", DATA_DIR.resolve())

## 1. Split, groups and images (identical to Notebook 05, but **no resize**)

In [ ]:
split = pd.read_csv("splits.csv")
SPLIT_MD5 = hashlib.md5(Path("splits.csv").read_bytes()).hexdigest()
GROUP = {"Cell": "Cell-level", "Cell-Multi": "Cell-level", "Cracking": "Cell-level",
         "Hot-Spot": "Cell-level", "Hot-Spot-Multi": "Cell-level",
         "Diode": "Substring-level", "Diode-Multi": "Substring-level",
         "Offline-Module": "Module-level",
         "Shadowing": "External", "Vegetation": "External", "Soiling": "External",
         "No-Anomaly": "No-Anomaly"}
CLASSES = sorted(split.label.unique())
c2i = {c: i for i, c in enumerate(CLASSES)}
Y = split.label.map(c2i).values
PARTS = {s: np.where(split.split.values == s)[0] for s in ("train", "val", "test")}

t0 = time.time()
imgs = np.stack([np.asarray(Image.open(DATA_DIR / p), dtype=np.float32) / 255.0 for p in split.path])
med = np.median(imgs, axis=(1, 2), keepdims=True)
std = imgs.std(axis=(1, 2), keepdims=True) + 1e-6
Z = torch.from_numpy((imgs - med) / std).unsqueeze(1)          # (20000, 1, 40, 24): this is ALL the preprocessing
print("images:", tuple(Z.shape), f"loaded in {time.time() - t0:.1f}s | splits.csv MD5 {SPLIT_MD5}")

In [ ]:
class IRModules(Dataset):
    # training only: flips, shift +-2 px (replicate padding, crop back to 40x24), optional Gaussian noise
    def __init__(self, idx, augment=False, max_shift=2, noise_std=0.0):
        self.idx = torch.as_tensor(idx); self.y = torch.as_tensor(Y[idx]).long()
        self.augment, self.s, self.noise = augment, max_shift, noise_std
    def __len__(self):
        return len(self.y)
    def __getitem__(self, i):
        x = Z[self.idx[i]]
        if self.augment:
            if torch.rand(1) < 0.5: x = torch.flip(x, dims=[2])
            if torch.rand(1) < 0.5: x = torch.flip(x, dims=[1])
            s = self.s
            if s > 0:
                x = F.pad(x.unsqueeze(0), (s, s, s, s), mode="replicate").squeeze(0)
                dy, dx = torch.randint(0, 2 * s + 1, (2,)).tolist()
                x = x[:, dy:dy + 40, dx:dx + 24]
            if self.noise > 0:
                x = x + self.noise * torch.randn_like(x)
        return x, self.y[i]

def make_loaders(noise_std=0.0, batch=128, seed=42):
    tr = PARTS["train"]
    counts = np.bincount(Y[tr], minlength=len(CLASSES))
    w = counts[Y[tr]] ** -0.5                                         # sampler p ~ 1/sqrt(n), as R10
    g = torch.Generator().manual_seed(seed)
    sampler = WeightedRandomSampler(torch.as_tensor(w, dtype=torch.double), num_samples=len(tr),
                                    replacement=True, generator=g)
    train_dl = DataLoader(IRModules(tr, augment=True, noise_std=noise_std), batch_size=batch, sampler=sampler)
    # fixed 3,000 training images, no augmentation, natural class mix: measures the training loss on the
    # same terms as the validation loss, so the train/validation gap is a fair overfitting measure
    rng = np.random.default_rng(0)
    tr_eval = DataLoader(IRModules(rng.choice(tr, 3000, replace=False)), batch_size=512)
    val_dl = DataLoader(IRModules(PARTS["val"]), batch_size=512)
    test_dl = DataLoader(IRModules(PARTS["test"]), batch_size=512)
    return train_dl, tr_eval, val_dl, test_dl, counts

## 2. The six architectures
All of them take the 40 × 24 × 1 image directly. The comments give the size of the feature map after each stage.

In [ ]:
class ResNet18S(nn.Module):
    # ResNet-18 adapted to small images (He et al. 2016, CIFAR-10 setting): conv1 3x3 stride 1, no max-pool
    # 40x24 -> layer1 40x24 -> layer2 20x12 -> layer3 10x6 -> layer4 5x3 -> global avg-pool -> fc
    def __init__(self, n_classes, pretrained=True, dropout=0.0):
        super().__init__()
        net = resnet18(weights=ResNet18_Weights.IMAGENET1K_V1 if pretrained else None)
        w7 = net.conv1.weight.data.clone()                             # (64, 3, 7, 7) pretrained
        net.conv1 = nn.Conv2d(1, 64, kernel_size=3, stride=1, padding=1, bias=False)
        if pretrained:                                                 # centre 3x3 of each 7x7 filter, RGB summed
            net.conv1.weight.data.copy_(w7[:, :, 2:5, 2:5].sum(dim=1, keepdim=True))
        net.maxpool = nn.Identity()
        net.fc = nn.Sequential(nn.Dropout(dropout), nn.Linear(512, n_classes))
        self.net = net
    def forward(self, x):
        return self.net(x)

class WideBasic(nn.Module):
    # pre-activation block of Wide ResNet: BN-ReLU-conv3x3 -> (dropout) -> BN-ReLU-conv3x3, plus shortcut
    def __init__(self, c_in, c_out, stride, dropout):
        super().__init__()
        self.bn1, self.conv1 = nn.BatchNorm2d(c_in), nn.Conv2d(c_in, c_out, 3, stride, 1, bias=False)
        self.bn2, self.conv2 = nn.BatchNorm2d(c_out), nn.Conv2d(c_out, c_out, 3, 1, 1, bias=False)
        self.drop = nn.Dropout(dropout)
        self.short = None if (stride == 1 and c_in == c_out) else nn.Conv2d(c_in, c_out, 1, stride, bias=False)
    def forward(self, x):
        o = F.relu(self.bn1(x))
        sc = x if self.short is None else self.short(o)
        o = self.conv1(o)
        o = self.conv2(self.drop(F.relu(self.bn2(o))))
        return o + sc

class WRN(nn.Module):
    # Wide ResNet (Zagoruyko & Komodakis 2016). WRN-16-4: 3 groups x 2 blocks, widths 64/128/256
    # 40x24 -> group1 40x24 -> group2 20x12 -> group3 10x6 -> global avg-pool -> fc
    def __init__(self, n_classes, depth=16, k=4, dropout=0.0):
        super().__init__()
        n = (depth - 4) // 6
        widths = [16, 16 * k, 32 * k, 64 * k]
        self.conv = nn.Conv2d(1, widths[0], 3, 1, 1, bias=False)
        blocks, c_in = [], widths[0]
        for g, stride in enumerate([1, 2, 2]):
            for i in range(n):
                blocks.append(WideBasic(c_in, widths[g + 1], stride if i == 0 else 1, dropout))
                c_in = widths[g + 1]
        self.blocks = nn.Sequential(*blocks)
        self.bn = nn.BatchNorm2d(c_in)
        self.fc = nn.Linear(c_in, n_classes)
    def forward(self, x):
        o = F.relu(self.bn(self.blocks(self.conv(x))))
        return self.fc(F.adaptive_avg_pool2d(o, 1).flatten(1))

def conv_bn(c_in, c_out, k=3):
    return nn.Sequential(nn.Conv2d(c_in, c_out, k, padding=k // 2, bias=False), nn.BatchNorm2d(c_out), nn.ReLU())

class MSCNN(nn.Module):
    # multi-scale CNN, idea of Korkmaz & Acikgoz 2022 (three branches with different kernel sizes);
    # simplified re-implementation. branches 3x3 / 5x5 / 7x7 at 40x24 -> concat 96 ch
    # -> 128 ch, pool 20x12 -> 128 ch, pool 10x6 -> 256 ch -> global avg-pool -> fc
    def __init__(self, n_classes, dropout=0.0):
        super().__init__()
        self.branches = nn.ModuleList([nn.Sequential(conv_bn(1, 32, k), conv_bn(32, 32, k)) for k in (3, 5, 7)])
        self.body = nn.Sequential(conv_bn(96, 128), nn.MaxPool2d(2), conv_bn(128, 128), nn.MaxPool2d(2),
                                  conv_bn(128, 256))
        self.head = nn.Sequential(nn.Dropout(dropout), nn.Linear(256, n_classes))
    def forward(self, x):
        o = torch.cat([b(x) for b in self.branches], dim=1)
        o = self.body(o)
        return self.head(F.adaptive_avg_pool2d(o, 1).flatten(1))

class CifarBasic(nn.Module):
    # basic residual block of the CIFAR ResNets (He et al. 2016)
    def __init__(self, c_in, c_out, stride):
        super().__init__()
        self.c1, self.b1 = nn.Conv2d(c_in, c_out, 3, stride, 1, bias=False), nn.BatchNorm2d(c_out)
        self.c2, self.b2 = nn.Conv2d(c_out, c_out, 3, 1, 1, bias=False), nn.BatchNorm2d(c_out)
        self.short = (nn.Identity() if (stride == 1 and c_in == c_out) else
                      nn.Sequential(nn.Conv2d(c_in, c_out, 1, stride, bias=False), nn.BatchNorm2d(c_out)))
    def forward(self, x):
        o = F.relu(self.b1(self.c1(x)))
        return F.relu(self.b2(self.c2(o)) + self.short(x))

class ResNetCifar(nn.Module):
    # ResNet-20 for small images (He et al. 2016, CIFAR-10): 3 stages x 3 blocks, widths 16/32/64
    # 40x24 -> stage1 40x24 -> stage2 20x12 -> stage3 10x6 -> global avg-pool -> fc
    def __init__(self, n_classes, depth=20, dropout=0.0):
        super().__init__()
        n = (depth - 2) // 6
        self.stem = nn.Sequential(nn.Conv2d(1, 16, 3, 1, 1, bias=False), nn.BatchNorm2d(16), nn.ReLU())
        layers, c_in = [], 16
        for c_out, stride in [(16, 1), (32, 2), (64, 2)]:
            for i in range(n):
                layers.append(CifarBasic(c_in, c_out, stride if i == 0 else 1)); c_in = c_out
        self.layers = nn.Sequential(*layers)
        self.head = nn.Sequential(nn.Dropout(dropout), nn.Linear(64, n_classes))
    def forward(self, x):
        o = self.layers(self.stem(x))
        return self.head(F.adaptive_avg_pool2d(o, 1).flatten(1))

class DenseLayer(nn.Module):
    # DenseNet-BC bottleneck layer: BN-ReLU-conv1x1 (4k) -> BN-ReLU-conv3x3 (k), output concatenated to its input
    def __init__(self, c_in, k):
        super().__init__()
        self.f = nn.Sequential(nn.BatchNorm2d(c_in), nn.ReLU(), nn.Conv2d(c_in, 4 * k, 1, bias=False),
                               nn.BatchNorm2d(4 * k), nn.ReLU(), nn.Conv2d(4 * k, k, 3, 1, 1, bias=False))
    def forward(self, x):
        return torch.cat([x, self.f(x)], dim=1)

class DenseNetBC(nn.Module):
    # DenseNet-BC for small images (Huang et al. 2017), L = 100, k = 12, compression 0.5
    # 40x24 -> block1 40x24 -> transition 20x12 -> block2 -> transition 10x6 -> block3 -> global avg-pool -> fc
    def __init__(self, n_classes, depth=100, k=12, dropout=0.0):
        super().__init__()
        n = (depth - 4) // 6
        c = 2 * k
        mods = [nn.Conv2d(1, c, 3, 1, 1, bias=False)]
        for b in range(3):
            for _ in range(n):
                mods.append(DenseLayer(c, k)); c += k
            if b < 2:
                c_out = c // 2
                mods += [nn.BatchNorm2d(c), nn.ReLU(), nn.Conv2d(c, c_out, 1, bias=False), nn.AvgPool2d(2)]
                c = c_out
        mods += [nn.BatchNorm2d(c), nn.ReLU()]
        self.features = nn.Sequential(*mods)
        self.head = nn.Sequential(nn.Dropout(dropout), nn.Linear(c, n_classes))
    def forward(self, x):
        return self.head(F.adaptive_avg_pool2d(self.features(x), 1).flatten(1))

def build_model(name, n_classes, dropout):
    return {"ResNet18S": lambda: ResNet18S(n_classes, pretrained=True, dropout=dropout),
            "ResNet18S_scratch": lambda: ResNet18S(n_classes, pretrained=False, dropout=dropout),
            "WRN16_4": lambda: WRN(n_classes, 16, 4, dropout=dropout),
            "MSCNN": lambda: MSCNN(n_classes, dropout=dropout),
            "ResNet20": lambda: ResNetCifar(n_classes, 20, dropout=dropout),
            "DenseNetBC100": lambda: DenseNetBC(n_classes, 100, 12, dropout=dropout)}[name]()

MODEL_INFO = {
    "ResNet18S": "ResNet-18, small-image stem (conv1 3x3 s1, no max-pool; He et al. 2016 CIFAR setting), 1-channel 40x24 input, "
                 "layer1-4 ImageNet-pretrained (IMAGENET1K_V1), conv1 from centre 3x3 of pretrained 7x7 (RGB summed), all layers fine-tuned",
    "ResNet18S_scratch": "ResNet-18, small-image stem (conv1 3x3 s1, no max-pool), 1-channel 40x24 input, random initialisation",
    "WRN16_4": "Wide ResNet-16-4 (Zagoruyko & Komodakis 2016), pre-activation blocks, widths 64/128/256, 1-channel 40x24 input, random initialisation",
    "ResNet20": "ResNet-20 for small images (He et al. 2016, CIFAR-10): 3 stages x 3 basic blocks, widths 16/32/64, 1-channel 40x24 input, random initialisation",
    "DenseNetBC100": "DenseNet-BC L=100 k=12 for small images (Huang et al. 2017), compression 0.5, 1-channel 40x24 input, random initialisation",
    "MSCNN": "Multi-scale CNN (idea of Korkmaz & Acikgoz 2022, simplified): branches 3x3/5x5/7x7 (32 ch each) -> 128 -> pool -> 128 -> pool -> 256 -> GAP -> fc, random initialisation",
}
LR = {"ResNet18S": 3e-4, "ResNet18S_scratch": 1e-3, "WRN16_4": 1e-3, "MSCNN": 1e-3, "ResNet20": 1e-3, "DenseNetBC100": 1e-3}       # pretrained: smaller steps
EPOCHS = {"ResNet18S": 30, "ResNet18S_scratch": 40, "WRN16_4": 40, "MSCNN": 40, "ResNet20": 40, "DenseNetBC100": 40}           # from scratch: more epochs
RECIPES = {
    "base": dict(label_smoothing=0.0, dropout=0.0, weight_decay=1e-4, noise_std=0.0),
    "reg":  dict(label_smoothing=0.1, dropout=0.3, weight_decay=5e-4, noise_std=0.05),
}

# self-test: shapes, parameters and one training step per model (nothing is logged)
for name in MODEL_INFO:
    m = build_model(name, 12, 0.3).to(device)
    n_par = sum(p.numel() for p in m.parameters() if p.requires_grad)
    x = Z[:8].to(device); y = torch.as_tensor(Y[:8]).to(device)
    opt = torch.optim.AdamW(m.parameters(), lr=1e-4)
    loss = F.cross_entropy(m.train()(x), y); loss.backward(); opt.step()
    with torch.no_grad():
        out = m.eval()(x)
    print(f"{name:18s} parameters = {n_par:>10,}   input {tuple(x.shape)} -> output {tuple(out.shape)}   one step OK")
del m, opt

## 3. One function that trains, tests and logs a run
Same as Notebook 05, plus:
* the **training-subset loss** (3,000 fixed training images, no augmentation) is computed every epoch with **plain cross-entropy**, like the validation loss (label smoothing raises the training loss on purpose, so the smoothed loss is not used for the comparison);
* new overfitting numbers per run: lowest validation loss and its epoch, validation loss at the last epoch, **rise** = last − lowest (0 means no rise), and **gap** = validation loss − training-subset loss at the last epoch.

In [ ]:
def environment():
    return {"python": sys.version.split()[0], "pytorch": torch.__version__, "torchvision": torchvision.__version__,
            "cuda": torch.version.cuda, "cudnn": torch.backends.cudnn.version() if torch.backends.cudnn.is_available() else None,
            "device": str(device), "hardware": HARDWARE, "os": platform.platform(),
            "data_dir": str(DATA_DIR.resolve()), "splits_md5": SPLIT_MD5}

def logged_ids():
    ids = set()
    for f in ("results_05.csv", "results_06.csv"):
        if Path(f).exists():
            ids |= set(pd.read_csv(f).run_id.astype(str))
    if Path("experiment_log.xlsx").exists():
        ws = load_workbook("experiment_log.xlsx", read_only=True)["Runs"]
        ids |= {str(r[0]) for r in ws.iter_rows(min_row=2, values_only=True) if r and r[0]}
    return ids

def next_free_id(run_id):
    ids = logged_ids()
    if run_id not in ids:
        return run_id
    k = 1
    while f"{run_id}-rerun{k}" in ids:
        k += 1
    return f"{run_id}-rerun{k}"

def evaluate(model, dl):
    model.eval(); ys, ps, loss_sum = [], [], 0.0
    with torch.no_grad():
        for x, y in dl:
            out = model(x.to(device))
            loss_sum += F.cross_entropy(out, y.to(device), reduction="sum").item()      # plain CE, no smoothing
            ps.append(out.argmax(1).cpu()); ys.append(y)
    yt, pt = torch.cat(ys).numpy(), torch.cat(ps).numpy()
    return yt, pt, loss_sum / len(yt)

def _replace_rows(ws, run_id, rows):
    for r in range(ws.max_row, 1, -1):
        if ws.cell(r, 1).value == run_id:
            ws.delete_rows(r)
    style_row = ws.max_row
    for values in rows:
        ws.append(values); r = ws.max_row
        for c in range(1, len(values) + 1):
            src, dst = ws.cell(style_row, c), ws.cell(r, c)
            if src.has_style:
                dst.font, dst.border, dst.alignment = copy(src.font), copy(src.border), copy(src.alignment)
                dst.number_format = src.number_format

def log_run(res, params, note=""):
    path = Path("experiment_log.xlsx")
    if not path.exists():
        print("experiment_log.xlsx not found: results not logged"); return
    wb = load_workbook(path)
    _replace_rows(wb["Runs"], res["run_id"], [[
        res["run_id"], datetime.date.today().isoformat(), NOTEBOOK, res["model"] + f" [{res['recipe']}]",
        "Raptor Maps, 12 classes", "splits.csv (70/15/15, seed 42)", "Raw pixels, per-image z-score, native 40x24 (no resize)",
        round(res["val_acc"], 3), round(res["val_f1"], 3), round(res["test_acc"], 3), round(res["test_f1"], 3),
        round(res["fault_acc"], 3), round(res["train_time"], 1), HARDWARE,
        (note + "; " if note else "") + f"best epoch {res['best_epoch']}; min val loss {res['val_loss_min']:.3f} @ ep {res['val_loss_min_epoch']}"]])
    _replace_rows(wb["Parameters"], res["run_id"], [[res["run_id"], "Run", k, v] for k, v in params.items()])
    wb.save(path)
    print(f"{res['run_id']} logged in experiment_log.xlsx")

def run_experiment(run_id, model_name, recipe, seed=42, batch=128, max_shift=2, note=""):
    rc = RECIPES[recipe]; lr, epochs = LR[model_name], EPOCHS[model_name]
    torch.manual_seed(seed); np.random.seed(seed)
    rerun_of = None
    free_id = next_free_id(run_id)
    if free_id != run_id:
        rerun_of, run_id = run_id, free_id
        note = f"Re-run of {rerun_of} with identical settings (original kept)" + (f"; {note}" if note else "")
    started = datetime.datetime.now()
    run_dir = Path("runs") / f"{run_id}_{started:%Y%m%d-%H%M%S}"
    run_dir.mkdir(parents=True, exist_ok=True)
    logf = open(run_dir / "train.log", "w", encoding="utf-8")
    def log(msg=""):
        line = f"[{datetime.datetime.now():%H:%M:%S}] {msg}"
        print(line); logf.write(line + "\n"); logf.flush()

    train_dl, tr_eval, val_dl, test_dl, counts = make_loaders(rc["noise_std"], batch, seed)
    train_dl.dataset.s = max_shift
    model = build_model(model_name, len(CLASSES), rc["dropout"]).to(device)
    n_par = sum(p.numel() for p in model.parameters() if p.requires_grad)
    criterion = nn.CrossEntropyLoss(label_smoothing=rc["label_smoothing"])
    optimizer = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=rc["weight_decay"])
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=epochs)
    ckpt = Path("checkpoints") / f"{run_id}_{model_name}_{recipe}.pt"

    log(f"===== {run_id}: {model_name}, recipe = {recipe}, seed {seed}, input 1x40x24 (no resize) =====")
    log(f"folder {run_dir} | {n_par:,} parameters | lr {lr:g} | epochs {epochs} | batch {batch} | {rc} | {HARDWARE}")
    hist, best, best_ep, best_acc, t0 = [], -1.0, 0, 0.0, time.time()
    for ep in range(1, epochs + 1):
        model.train(); total, correct, seen = 0.0, 0, 0
        lr_now = optimizer.param_groups[0]["lr"]
        for x, y in train_dl:
            x, y = x.to(device), y.to(device)
            optimizer.zero_grad()
            out = model(x); loss = criterion(out, y)
            loss.backward(); optimizer.step()
            total += loss.item() * len(y); correct += (out.argmax(1) == y).sum().item(); seen += len(y)
        scheduler.step()
        ye, pe, tr_loss = evaluate(model, tr_eval)
        yv, pv, val_loss = evaluate(model, val_dl)
        f1v, accv = f1_score(yv, pv, average="macro"), accuracy_score(yv, pv)
        hist.append({"epoch": ep, "lr": lr_now, "train_loss_batches": total / seen, "train_acc_batches": correct / seen,
                     "train_eval_loss": tr_loss, "train_eval_acc": accuracy_score(ye, pe),
                     "val_loss": val_loss, "val_acc": accv, "val_macroF1": f1v, "seconds": round(time.time() - t0, 1)})
        if f1v > best:
            best, best_ep, best_acc = f1v, ep, accv
            torch.save(model.state_dict(), ckpt)
        log(f"epoch {ep:2d} | lr {lr_now:.2e} | train-subset loss {tr_loss:.3f} | val loss {val_loss:.3f} acc {accv:.3f} "
            f"macro-F1 {f1v:.3f}{' *' if ep == best_ep else ''} | {time.time() - t0:.0f}s")
    train_time = time.time() - t0
    h = pd.DataFrame(hist); h.to_csv(run_dir / "history.csv", index=False)

    model.load_state_dict(torch.load(ckpt, map_location=device))
    yt, pt, test_loss = evaluate(model, test_dl)
    names = np.array(CLASSES)
    i_min = int(h.val_loss.idxmin())
    res = dict(run_id=run_id, rerun_of=rerun_of, model=model_name, recipe=recipe, seed=seed,
               val_acc=best_acc, val_f1=best, best_epoch=best_ep,
               test_acc=accuracy_score(yt, pt), test_f1=f1_score(yt, pt, average="macro"),
               test_f1_as_5=f1_score([GROUP[c] for c in names[yt]], [GROUP[c] for c in names[pt]], average="macro"),
               fault_acc=float(np.mean((names[yt] == "No-Anomaly") == (names[pt] == "No-Anomaly"))),
               test_loss=test_loss,
               val_loss_min=float(h.val_loss.min()), val_loss_min_epoch=int(h.epoch[i_min]),
               val_loss_last=float(h.val_loss.iloc[-1]), val_loss_at_best=float(h.val_loss[h.epoch == best_ep].iloc[0]),
               val_loss_rise=float(h.val_loss.iloc[-1] - h.val_loss.min()),
               gap_last=float(h.val_loss.iloc[-1] - h.train_eval_loss.iloc[-1]),
               train_time=train_time, params=n_par)
    p, r, f, s = precision_recall_fscore_support(yt, pt, labels=range(len(CLASSES)), zero_division=0)
    per_class = pd.DataFrame({"class": CLASSES, "precision": p, "recall": r, "f1": f, "test_images": s}).round(3)
    per_class.to_csv(run_dir / "test_per_class.csv", index=False)
    cm_counts = confusion_matrix(yt, pt, labels=range(len(CLASSES)))
    pd.DataFrame(cm_counts, index=[f"true {c}" for c in CLASSES], columns=[f"pred {c}" for c in CLASSES]).to_csv(run_dir / "confusion_matrix.csv")
    pd.DataFrame({"path": split.path.values[PARTS["test"]], "true": names[yt], "pred": names[pt]}).to_csv(run_dir / "test_predictions.csv", index=False)

    log(f"best epoch {best_ep} | val macro-F1 {best:.3f} | lowest val loss {res['val_loss_min']:.3f} @ epoch {res['val_loss_min_epoch']} "
        f"| last val loss {res['val_loss_last']:.3f} (rise {res['val_loss_rise']:.3f}, gap {res['gap_last']:.3f}) | {train_time:.0f}s")
    log(f"TEST accuracy {res['test_acc']:.3f} | macro-F1 {res['test_f1']:.3f} | as 5 groups {res['test_f1_as_5']:.3f} | fault/no-fault {res['fault_acc']:.3f}")

    fig, ax = plt.subplots(1, 3, figsize=(16, 4.5), gridspec_kw={"width_ratios": [1, 1, 1.4]})
    ax[0].plot(h.epoch, h.train_eval_loss, color="#2E5E8C", lw=2, label="train subset")
    ax[0].plot(h.epoch, h.val_loss, color="#8A1F2B", lw=2, label="validation")
    ax[0].axvline(res["val_loss_min_epoch"], color="#888", ls=":", lw=1)
    ax[0].set_title("Loss (plain cross-entropy)"); ax[0].set_xlabel("epoch"); ax[0].legend(frameon=False)
    ax[1].plot(h.epoch, h.val_macroF1, color="#8A1F2B", lw=2); ax[1].axvline(best_ep, color="#888", ls="--", lw=1)
    ax[1].set_title("Validation macro-F1"); ax[1].set_xlabel("epoch")
    cm = confusion_matrix(yt, pt, labels=range(len(CLASSES)), normalize="true")
    ConfusionMatrixDisplay(cm, display_labels=CLASSES).plot(ax=ax[2], cmap="Reds", values_format=".2f", colorbar=False)
    ax[2].tick_params(axis="x", rotation=60); ax[2].set_title(f"{run_id}: test confusion matrix")
    for a in ax[:2]: a.grid(alpha=0.25)
    plt.tight_layout(); plt.savefig(run_dir / "curves_and_cm.png", dpi=200)
    plt.savefig(f"figures/{run_id}_{model_name}_{recipe}.png", dpi=200); plt.close(fig)

    config = {"run_id": run_id, "rerun_of": rerun_of, "note": note, "started": started.isoformat(timespec="seconds"),
              "notebook": NOTEBOOK, "run_folder": str(run_dir), "checkpoint": str(ckpt),
              "data": {"dataset": "Raptor Maps InfraredSolarModules (2020-02-14)", "split_file": "splits.csv", "classes": CLASSES,
                       "train_images_per_class": dict(zip(CLASSES, counts.tolist()))},
              "input": "1 x 40 x 24, per-image z = (I - median) / (std + 1e-6), NO resize, NO channel repeat",
              "augmentation": {"hflip_p": 0.5, "vflip_p": 0.5, "random_shift_px": max_shift, "shift_padding": "replicate",
                               "gaussian_noise_std": rc["noise_std"], "applied_to": "training set only"},
              "balancing": "WeightedRandomSampler, p = 1/sqrt(n_class), 14,000 draws per epoch, replacement",
              "model": {"name": model_name, "description": MODEL_INFO[model_name], "trainable_parameters": n_par},
              "training": {"recipe": recipe, "loss": f"CrossEntropyLoss(label_smoothing={rc['label_smoothing']})",
                           "dropout": rc["dropout"], "optimizer": "AdamW", "lr": lr, "weight_decay": rc["weight_decay"],
                           "lr_schedule": f"CosineAnnealingLR(T_max={epochs})", "epochs": epochs, "batch_train": batch,
                           "batch_eval": 512, "seed": seed, "model_selection": "best validation macro-F1",
                           "overfit_monitor": "plain CE on 3,000 fixed training images (no augmentation) and on validation"},
              "environment": environment(),
              "results": {k: (round(float(v), 4) if isinstance(v, (float, np.floating)) else v) for k, v in res.items()}}
    config["results"]["finished"] = datetime.datetime.now().isoformat(timespec="seconds")
    (run_dir / "config.json").write_text(json.dumps(config, indent=2, ensure_ascii=False, default=str), encoding="utf-8")

    params = {
        "Run folder": str(run_dir), "Classes": f"12 ({', '.join(CLASSES)})",
        "Input": "1 x 40 x 24 native, per-image z-score; no resize, no channel repeat",
        "Balancing": "WeightedRandomSampler, p ∝ 1/sqrt(n_class), 14,000 draws per epoch, replacement",
        "Augmentation": f"H flip p=0.5, V flip p=0.5, shift ±{max_shift} px (replicate), Gaussian noise σ={rc['noise_std']}; training only",
        "Model": MODEL_INFO[model_name], "Trainable parameters": f"{n_par:,}",
        "Recipe": recipe, "Loss": f"Cross-entropy, label smoothing {rc['label_smoothing']}", "Dropout": str(rc["dropout"]),
        "Optimizer": f"AdamW, lr = {lr:g}, weight_decay = {rc['weight_decay']:g}",
        "LR schedule": f"CosineAnnealingLR, T_max = {epochs}", "Epochs / batch": f"{epochs} / {batch} (eval batch 512)",
        "Model selection": f"best validation macro-F1 → epoch {best_ep}",
        "Overfitting": (f"min val loss {res['val_loss_min']:.3f} @ epoch {res['val_loss_min_epoch']}; last {res['val_loss_last']:.3f}; "
                        f"rise {res['val_loss_rise']:.3f}; gap (val − train subset) {res['gap_last']:.3f}"),
        "Random seed": f"{seed} (torch, numpy, sampler; data split always seed 42)", "splits.csv MD5": SPLIT_MD5,
        "Environment": f"Python {sys.version.split()[0]}, PyTorch {torch.__version__}, CUDA {torch.version.cuda}, {device} ({HARDWARE}), {platform.platform()}",
        "Test macro-F1 scored as 5 groups": f"{res['test_f1_as_5']:.3f}",
        "Test precision / recall / F1 per class": "; ".join(f"{row['class']} {row.precision}/{row.recall}/{row.f1}" for _, row in per_class.iterrows()),
    }
    log_run(res, params, note)
    log(f"saved: {run_dir}"); logf.close()
    return res

RES_COLS = ["run_id", "rerun_of", "model", "recipe", "seed", "val_f1", "best_epoch", "test_acc", "test_f1", "test_f1_as_5", "fault_acc",
            "test_loss", "val_loss_min", "val_loss_min_epoch", "val_loss_at_best", "val_loss_last", "val_loss_rise", "gap_last", "train_time", "params"]
def save_result(r):
    new = pd.DataFrame([r]).reindex(columns=RES_COLS)
    old = pd.read_csv("results_06.csv").reindex(columns=RES_COLS) if Path("results_06.csv").exists() else pd.DataFrame(columns=RES_COLS)
    pd.concat([old[~old.run_id.isin(new.run_id)], new], ignore_index=True).to_csv("results_06.csv", index=False)

## 4. Run plan (R26–R61)
| Recipe | ResNet18S | ResNet18S_scratch | WRN16_4 | MSCNN | ResNet20 | DenseNetBC100 |
|---|---|---|---|---|---|---|
| `base` | R26–R28 | R29–R31 | R32–R34 | R35–R37 | R38–R40 | R41–R43 |
| `reg` | R44–R46 | R47–R49 | R50–R52 | R53–R55 | R56–R58 | R59–R61 |

Seeds 42, 1, 2 in that order. To run only part of it, edit `ONLY` (e.g. `ONLY = ["R26", "R38"]`) or leave it `None` for all.
Results are saved after every run, so an interruption loses nothing.

In [ ]:
MODELS = ["ResNet18S", "ResNet18S_scratch", "WRN16_4", "MSCNN", "ResNet20", "DenseNetBC100"]
PLAN, n = [], 26
for recipe in ("base", "reg"):
    for m in MODELS:
        for sd in (42, 1, 2):
            PLAN.append((f"R{n}", m, recipe, sd)); n += 1
ONLY = None

results = []
t_all = time.time()
for rid, m, recipe, sd in PLAN:
    if ONLY is not None and rid not in ONLY:
        continue
    r = run_experiment(rid, m, recipe, seed=sd,
                       note=f"Preserve input 40x24; {m}; recipe {recipe}; seed {sd} (advisor feedback 11 Nov 2026)")
    results.append(r); save_result(r)
print(f"all done in {(time.time() - t_all) / 60:.1f} min")

## 5. Mean ± SD over 3 seeds, with the Notebook 05 reference (R10, R16, R17)
`f1_5` = macro-F1 after merging predictions into the 5 groups. Lower `val_loss_min`, `rise` and `gap` = less overfitting.
For the reference runs the validation-loss numbers come from their `history.csv`; their training-subset loss was not recorded (gap = NaN).

In [ ]:
t = pd.read_csv("results_06.csv")
t = t[t.rerun_of.isna()] if "rerun_of" in t else t
summary = (t.groupby(["model", "recipe"])
             .agg(runs=("run_id", lambda x: ", ".join(x)), n=("run_id", "size"), params=("params", "first"),
                  acc=("test_acc", "mean"), f1=("test_f1", "mean"), f1_sd=("test_f1", "std"),
                  f1_5=("test_f1_as_5", "mean"), f1_5_sd=("test_f1_as_5", "std"), fault=("fault_acc", "mean"),
                  val_loss_min=("val_loss_min", "mean"), val_loss_last=("val_loss_last", "mean"),
                  rise=("val_loss_rise", "mean"), gap=("gap_last", "mean"), time_s=("train_time", "mean"))
             .reset_index())

ref_rows = []
if Path("results_05.csv").exists():
    r5 = pd.read_csv("results_05.csv")
    r5 = r5[r5.run_id.isin(["R10", "R16", "R17"])]
    for _, row in r5.iterrows():
        hp = sorted(Path("runs").glob(f"{row.run_id}_*/history.csv"))
        vmin = vlast = np.nan
        if hp:
            hh = pd.read_csv(hp[-1]); vmin, vlast = hh.val_loss.min(), hh.val_loss.iloc[-1]
        ref_rows.append(dict(run_id=row.run_id, acc=row.test_acc, f1=row.test_f1, f1_5=row.test_f1_as_5, fault=row.fault_acc,
                             val_loss_min=vmin, val_loss_last=vlast, time_s=row.train_time, params=row.params))
if ref_rows:
    rf = pd.DataFrame(ref_rows)
    ref = dict(model="ResNet18TL (resize 160x96, NB05)", recipe="base", runs=", ".join(rf.run_id), n=len(rf), params=rf.params.iloc[0],
               acc=rf.acc.mean(), f1=rf.f1.mean(), f1_sd=rf.f1.std(), f1_5=rf.f1_5.mean(), f1_5_sd=rf.f1_5.std(), fault=rf.fault.mean(),
               val_loss_min=rf.val_loss_min.mean(), val_loss_last=rf.val_loss_last.mean(),
               rise=(rf.val_loss_last - rf.val_loss_min).mean(), gap=np.nan, time_s=rf.time_s.mean())
    summary = pd.concat([pd.DataFrame([ref]), summary], ignore_index=True)

summary = summary.round(3).sort_values("f1_5", ascending=False)
summary.to_csv("results_06_summary.csv", index=False)
print(summary.to_string(index=False))

## 6. Overfitting figures
* `figures/fig06_val_loss_all.png`: validation loss of every setup (seed 42), with R10 for reference.
* `figures/fig06_overfit_best.png`: training-subset vs validation loss of the best setup, `base` against `reg`.

In [ ]:
def hist_of(run_id):
    hp = sorted(Path("runs").glob(f"{run_id}_*/history.csv"))
    return pd.read_csv(hp[-1]) if hp else None

seed42 = t[t.seed == 42]
colors = {"ResNet18S": "#8A1F2B", "ResNet18S_scratch": "#D98C8C", "WRN16_4": "#2E5E8C", "MSCNN": "#C9871A", "ResNet20": "#3E9B6B", "DenseNetBC100": "#6B3FA0"}
fig, ax = plt.subplots(1, 2, figsize=(14, 4.5), sharey=True)
for k, recipe in enumerate(("base", "reg")):
    h10 = hist_of("R10")
    if h10 is not None:
        ax[k].plot(h10.epoch, h10.val_loss, color="#888", lw=2, ls="--", label="R10 (resize 160x96)")
    for _, row in seed42[seed42.recipe == recipe].iterrows():
        h = hist_of(row.run_id)
        if h is not None:
            ax[k].plot(h.epoch, h.val_loss, color=colors.get(row.model, "k"), lw=2, label=f"{row.run_id} {row.model}")
    ax[k].set_title(f"Validation loss, recipe '{recipe}' (seed 42)"); ax[k].set_xlabel("epoch"); ax[k].grid(alpha=.3); ax[k].legend(fontsize=8)
ax[0].set_ylabel("plain cross-entropy")
plt.tight_layout(); plt.savefig("figures/fig06_val_loss_all.png", dpi=200); plt.show()

best_model = summary[~summary.model.str.startswith("ResNet18TL")].iloc[0].model
fig, ax = plt.subplots(1, 2, figsize=(14, 4.5), sharey=True)
for k, recipe in enumerate(("base", "reg")):
    row = seed42[(seed42.model == best_model) & (seed42.recipe == recipe)]
    if len(row):
        h = hist_of(row.run_id.iloc[0])
        ax[k].plot(h.epoch, h.train_eval_loss, color="#2E5E8C", lw=2, label="train subset")
        ax[k].plot(h.epoch, h.val_loss, color="#8A1F2B", lw=2, label="validation")
        ax[k].fill_between(h.epoch, h.train_eval_loss, h.val_loss, color="#8A1F2B", alpha=.08)
        ax[k].set_title(f"{best_model}, recipe '{recipe}' ({row.run_id.iloc[0]})")
    ax[k].set_xlabel("epoch"); ax[k].grid(alpha=.3); ax[k].legend()
ax[0].set_ylabel("plain cross-entropy")
plt.tight_layout(); plt.savefig("figures/fig06_overfit_best.png", dpi=200); plt.show()

## 7. How to read the results
1. **Preserve input vs resize:** compare every `ResNet18S` row with the reference `ResNet18TL (resize 160x96)`. Same backbone, only the input handling differs.
2. **Does ImageNet still help with the new stem?** `ResNet18S` vs `ResNet18S_scratch`.
3. **Which architecture for small images?** `ResNet18S` vs `WRN16_4` vs `MSCNN` vs `ResNet20` vs `DenseNetBC100` (same recipe). Is a smaller network (ResNet20, DenseNet) less overfitted?
4. **Overfitting:** for each model, `base` vs `reg`: is `val_loss_min` lower, `rise` closer to 0 and `gap` smaller, **without** a lower `f1_5`?
5. **Decision rule (set before the results):** prefer the setup with the highest mean `f1_5`; if two are within about one SD (≈ 0.01), prefer the one with the lower validation loss, then the faster one.